# Monthyl Core HR metrics

In [1]:
import pandas as pd
import numpy as np

## Data preparation

In [2]:
fields = [
    'ds_start', 'ds', 'employee_id', 'hire_date', 'prehire_status', 'is_active',
    'termination_date', 'termination_reason', 'is_termination_voluntary', 'is_terminated',
    'org_l00', 'org_l01', 'org_l02', 'org_l03', 
    'gender', 'gender_remapped', 'ethnicity', 'ethnicity_remapped', 
    'is_manager_track', 'job_track', 'job_level_idx',
    'job_level_category', 'job_level_category_ordered_w_indicators'
]

In [3]:
df_employees_plus = pd.read_csv(
    filepath_or_buffer='../transforms/exclude/employee_data_plus.csv',
    dtype='str'
)
# type casting and renaming fields
df_employees_plus['ds'] = pd.to_datetime(df_employees_plus['full_date']).dt.normalize()
df_employees_plus['ds_start'] = df_employees_plus.ds.dt.to_period('M').dt.start_time
df_employees_plus['hire_date'] = pd.to_datetime(df_employees_plus['hire_date']).dt.normalize()
df_employees_plus['termination_date'] = pd.to_datetime(df_employees_plus['termination_date_coalesced']).dt.normalize()
df_employees_plus['is_termination_voluntary'] = df_employees_plus.is_termination_voluntary.astype('bool')
df_employees_plus['is_manager_track'] = df_employees_plus.is_manager_track.astype('bool')

# deriving is_active
condition_active = (df_employees_plus.ds >= df_employees_plus.hire_date) &\
    (df_employees_plus.ds <= df_employees_plus.termination_date)
df_employees_plus['is_active'] = condition_active

# deriving is_terminated
condition_terminated_in_current_month = (df_employees_plus.ds >= df_employees_plus.hire_date) &\
    (df_employees_plus.ds_start <= df_employees_plus.termination_date) &\
    (df_employees_plus.ds >= df_employees_plus.termination_date)
df_employees_plus['is_terminated'] = condition_terminated_in_current_month

# reorganizing fields
df_employees_plus = df_employees_plus[fields]

df_employees_plus.dtypes

ds_start                                   datetime64[ns]
ds                                         datetime64[ns]
employee_id                                        object
hire_date                                  datetime64[ns]
prehire_status                                     object
is_active                                            bool
termination_date                           datetime64[ns]
termination_reason                                 object
is_termination_voluntary                             bool
is_terminated                                        bool
org_l00                                            object
org_l01                                            object
org_l02                                            object
org_l03                                            object
gender                                             object
gender_remapped                                    object
ethnicity                                          object
ethnicity_rema

In [4]:
df_employees_plus.head()

,ds_start,ds,employee_id,hire_date,prehire_status,is_active,termination_date,termination_reason,is_termination_voluntary,is_terminated,...,org_l03,gender,gender_remapped,ethnicity,ethnicity_remapped,is_manager_track,job_track,job_level_idx,job_level_category,job_level_category_ordered_w_indicators
0,2021-01-01,2021-01-31,e000001,2014-01-02,Not prehire,True,2260-01-01,NaN,True,False,...,NaN,Male,Male,White,00--White,True,M,11,SVP,10--SVP (M11)
1,2021-02-01,2021-02-28,e000001,2014-01-02,Not prehire,True,2260-01-01,NaN,True,False,...,NaN,Male,Male,White,00--White,True,M,11,SVP,10--SVP (M11)
2,2021-03-01,2021-03-31,e000001,2014-01-02,Not prehire,True,2260-01-01,NaN,True,False,...,NaN,Male,Male,White,00--White,True,M,11,SVP,10--SVP (M11)
3,2021-04-01,2021-04-30,e000001,2014-01-02,Not prehire,True,2260-01-01,NaN,True,False,...,NaN,Male,Male,White,00--White,True,M,11,SVP,10--SVP (M11)
4,2021-05-01,2021-05-31,e000001,2014-01-02,Not prehire,True,2260-01-01,NaN,True,False,...,NaN,Male,Male,White,00--White,True,M,11,SVP,10--SVP (M11)


## Monthly results

### Creating monthly dataframe

In [5]:
df_monthly = pd.DataFrame({'ds': df_employees_plus.ds.unique()})

### Deriving `n_active_employees`

Definitions: 
- Unique count of `employee_id`
- `hire_date < ds < termination_date`

In [6]:
mask_active_by_dates = (df_employees_plus.ds >= df_employees_plus.hire_date) &\
    (df_employees_plus.ds <= df_employees_plus.termination_date)
active_by_dates = df_employees_plus[mask_active_by_dates] \
    .groupby('ds')['employee_id'].nunique() \
    .reset_index(name='n_active_employees')


In [7]:
df_monthly = df_monthly.merge(right=active_by_dates, how='left', on='ds')

### Deriving `n_active_by_status`

In [8]:
# df_monthly['n_active_employees_by_status']
active_employees_by_status = df_employees_plus[df_employees_plus['is_active']] \
    .groupby('ds')['employee_id'].nunique() \
    .reset_index(name='active_employees_by_status')
df_monthly = df_monthly.merge(right=active_employees_by_status, how='left', on='ds')

### Deriving `n_monthly_terminated_employees`

In [9]:
mask_terminated_by_month = (df_employees_plus.ds >= df_employees_plus.hire_date) &\
    (df_employees_plus.ds_start <= df_employees_plus.termination_date) &\
    (df_employees_plus.ds >= df_employees_plus.termination_date)
terminated_by_dates_monthly = df_employees_plus[mask_terminated_by_month] \
    .groupby('ds')['employee_id'].nunique() \
    .reset_index(name='n_monthly_terminated_employees')

df_monthly = df_monthly.merge(right=terminated_by_dates_monthly, how='left', on='ds')
df_monthly['n_monthly_terminated_employees'] = df_monthly.n_monthly_terminated_employees.fillna(0).astype('int')


### Deriving `n_monthly_terminated_employees_by_status`

In [10]:
terminated_by_dates_monthly_by_status = df_employees_plus[df_employees_plus.is_terminated] \
    .groupby('ds')['employee_id'].nunique() \
    .reset_index(name='n_monthly_terminated_employees_by_status')

df_monthly = df_monthly.merge(right=terminated_by_dates_monthly_by_status, how='left', on='ds')
df_monthly['n_monthly_terminated_employees_by_status'] = df_monthly.n_monthly_terminated_employees_by_status.fillna(0).astype('int')


### Deriving fields for `attrition_rate`

- `avg_active_employees` - active employee count 2 month rolling (between current month and previous month)
- `attrition_rate = terminated_employees / avg_active_employees`

In [11]:
df_monthly['avg_active_employees'] = df_monthly['n_active_employees'].rolling(window=2).mean()
df_monthly['avg_active_employees'] = np.where(
    df_monthly['avg_active_employees'].isnull(),
    df_monthly['n_active_employees'],
    df_monthly['avg_active_employees']
)
df_monthly['attrition_rate'] = df_monthly['n_monthly_terminated_employees'] / df_monthly['avg_active_employees']

## Monthly results by organization

In [12]:
# df_monthly_by_org = 
df_employees_plus

,ds_start,ds,employee_id,hire_date,prehire_status,is_active,termination_date,termination_reason,is_termination_voluntary,is_terminated,...,org_l03,gender,gender_remapped,ethnicity,ethnicity_remapped,is_manager_track,job_track,job_level_idx,job_level_category,job_level_category_ordered_w_indicators
0,2021-01-01,2021-01-31,e000001,2014-01-02,Not prehire,True,2260-01-01,NaN,True,False,...,NaN,Male,Male,White,00--White,True,M,11,SVP,10--SVP (M11)
1,2021-02-01,2021-02-28,e000001,2014-01-02,Not prehire,True,2260-01-01,NaN,True,False,...,NaN,Male,Male,White,00--White,True,M,11,SVP,10--SVP (M11)
2,2021-03-01,2021-03-31,e000001,2014-01-02,Not prehire,True,2260-01-01,NaN,True,False,...,NaN,Male,Male,White,00--White,True,M,11,SVP,10--SVP (M11)
3,2021-04-01,2021-04-30,e000001,2014-01-02,Not prehire,True,2260-01-01,NaN,True,False,...,NaN,Male,Male,White,00--White,True,M,11,SVP,10--SVP (M11)
4,2021-05-01,2021-05-31,e000001,2014-01-02,Not prehire,True,2260-01-01,NaN,True,False,...,NaN,Male,Male,White,00--White,True,M,11,SVP,10--SVP (M11)
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
412195,2026-08-01,2026-08-31,e005725,2024-11-25,Not prehire,True,2260-01-01,NaN,True,False,...,Consumer Direct,Male,Male,Hispanic,02--Minority,True,IC,3,Associate,02--Associate (IC1-3)
412196,2026-09-01,2026-09-30,e005725,2024-11-25,Not prehire,True,2260-01-01,NaN,True,False,...,Consumer Direct,Male,Male,Hispanic,02--Minority,True,IC,3,Associate,02--Associate (IC1-3)
412197,2026-10-01,2026-10-31,e005725,2024-11-25,Not prehire,True,2260-01-01,NaN,True,False,...,Consumer Direct,Male,Male,Hispanic,02--Minority,True,IC,3,Associate,02--Associate (IC1-3)
412198,2026-11-01,2026-11-30,e005725,2024-11-25,Not prehire,True,2260-01-01,NaN,True,False,...,Consumer Direct,Male,Male,Hispanic,02--Minority,True,IC,3,Associate,02--Associate (IC1-3)


In [13]:
df_employees_plus[['ds', 'ds_start', 'employee_id', 'hire_date', 'termination_date']][
    mask_terminated_by_month
    & (df_employees_plus.ds == '2024-11-30')
]

,ds,ds_start,employee_id,hire_date,termination_date
4942,2024-11-30,2024-11-01,e000069,2014-11-26,2024-11-22
35110,2024-11-30,2024-11-01,e000488,2017-03-24,2024-11-01
113878,2024-11-30,2024-11-01,e001582,2019-05-29,2024-11-01
124246,2024-11-30,2024-11-01,e001726,2019-08-22,2024-11-28
160678,2024-11-30,2024-11-01,e002232,2020-07-20,2024-11-07
162838,2024-11-30,2024-11-01,e002262,2020-07-31,2024-11-27
184582,2024-11-30,2024-11-01,e002564,2021-01-29,2024-11-18
217702,2024-11-30,2024-11-01,e003024,2022-02-24,2024-11-27
235486,2024-11-30,2024-11-01,e003271,2022-03-03,2024-11-26
249454,2024-11-30,2024-11-01,e003465,2022-06-07,2024-11-28
